In [44]:
import requests 
import pandas as pd

#will give 50 articles 
alphavantage = requests.get('https://www.alphavantage.co/query?function=NEWS_SENTIMENT&limit=10&apikey=J6SDVR8XMY3RQLMQ')

if alphavantage.status_code == 200:
    # Parse the response as JSON
    alphavantage_response = alphavantage.json()


In [45]:
df_2 = pd.DataFrame(alphavantage_response["feed"])

df_2.drop(['banner_image', 'authors', 'category_within_source', 'topics', 'source', 'overall_sentiment_label'], axis=1, inplace=True)

df_2.head()

,title,url,time_published,summary,source_domain,overall_sentiment_score,ticker_sentiment
0,Crystal Bridges Museum of American Art Awards ...,https://www.benzinga.com/pressreleases/24/12/g...,20241218T160000,"BENTONVILLE, Ark, Dec. 18, 2024 ( GLOBE NEWSWI...",www.benzinga.com,0.420169,"[{'ticker': 'TSN', 'relevance_score': '0.09706..."
1,Linda Itskovitz Joins Technology Credit Union'...,https://www.benzinga.com/pressreleases/24/12/g...,20241218T160000,"SAN JOSE, Calif., Dec. 18, 2024 ( GLOBE NEWSWI...",www.benzinga.com,0.404269,"[{'ticker': 'MTTR', 'relevance_score': '0.0936..."
2,ThoughtSpot Appoints Francois Lopitaux to Lead...,https://www.benzinga.com/pressreleases/24/12/g...,20241218T160000,"MOUNTAIN VIEW, Calif., Dec. 18, 2024 ( GLOBE N...",www.benzinga.com,0.389167,"[{'ticker': 'VRSK', 'relevance_score': '0.0476..."
3,What's Going On With Polyrizon Shares Wednesda...,https://www.benzinga.com/24/12/42557322/whats-...,20241218T155734,Polyrizon Ltd. PLRZ shares are moving higher a...,www.benzinga.com,0.359258,"[{'ticker': 'PLRZ', 'relevance_score': '0.9248..."
4,Discover the BRETT ( ETH ) Listing on XT,https://www.benzinga.com/pressreleases/24/12/g...,20241218T155254,"VICTORIA, Seychelles, Dec. 18, 2024 ( GLOBE NE...",www.benzinga.com,0.531296,"[{'ticker': 'CRYPTO:ETH', 'relevance_score': '..."


In [46]:
#concatentating all the topics into a list to put into "industry"
topics = []
for element in alphavantage_response["feed"]:
    templist = []
    for topic in element["topics"]:
        templist.append(topic["topic"])
    
    topics.append(templist)

df_2['industry'] = topics

In [47]:
tickers_2 = []
ticker_sentiment_2 = []

rowcount_2 = 0
for element in alphavantage_response["feed"]:
    if 'ticker_sentiment' in element and element['ticker_sentiment']:
        # Record ticker and sentiment scores
        for ticker in element["ticker_sentiment"]:
            tickers_2.append(ticker['ticker'])
            ticker_sentiment_2.append(ticker['ticker_sentiment_score'])
        
        # Duplicate rows as needed
        duplicatedrows = len(element['ticker_sentiment']) - 1
        if duplicatedrows > 0:
            new_row = df_2.iloc[rowcount_2].copy()
            df_2 = pd.concat([df_2, pd.DataFrame([new_row] * duplicatedrows)], ignore_index=True)
    
    else:
        tickers_2.append(None)
        ticker_sentiment_2.append(None)

    rowcount_2 += 1


# Ensure lengths match before assignment
if len(df_2) != len(tickers_2):
    raise ValueError(f"Row mismatch: DataFrame rows ({len(df_2)}) and tickers ({len(tickers_2)})")

# Assign tickers and sentiments
df_2['ticker'] = tickers_2
df_2['ticker_sentiment'] = ticker_sentiment_2


In [48]:
df_2.drop(df_2[df_2.title == 'Before you continue'].index, inplace=True)

In [49]:
df_2.sort_values('url')

,title,url,time_published,summary,source_domain,overall_sentiment_score,ticker_sentiment,industry,ticker
105,This Box Analyst Begins Coverage On A Bullish ...,https://www.benzinga.com/24/12/42555415/this-b...,20241218T150300,Top Wall Street analysts changed their outlook...,www.benzinga.com,0.283895,0.317181,"[Life Sciences, Financial Markets, Real Estate...",AAPL
104,This Box Analyst Begins Coverage On A Bullish ...,https://www.benzinga.com/24/12/42555415/this-b...,20241218T150300,Top Wall Street analysts changed their outlook...,www.benzinga.com,0.283895,0.269607,"[Life Sciences, Financial Markets, Real Estate...",RGTI
39,This Box Analyst Begins Coverage On A Bullish ...,https://www.benzinga.com/24/12/42555415/this-b...,20241218T150300,Top Wall Street analysts changed their outlook...,www.benzinga.com,0.283895,None,"[Life Sciences, Financial Markets, Real Estate...",None
103,This Box Analyst Begins Coverage On A Bullish ...,https://www.benzinga.com/24/12/42555415/this-b...,20241218T150300,Top Wall Street analysts changed their outlook...,www.benzinga.com,0.283895,0.416197,"[Life Sciences, Financial Markets, Real Estate...",META
102,This Box Analyst Begins Coverage On A Bullish ...,https://www.benzinga.com/24/12/42555415/this-b...,20241218T150300,Top Wall Street analysts changed their outlook...,www.benzinga.com,0.283895,0.244362,"[Life Sciences, Financial Markets, Real Estate...",AMZN
...,...,...,...,...,...,...,...,...,...
27,Davidson Kempner Capital Management LP : Form ...,https://www.globenewswire.com/news-release/202...,20241218T152000,FORM ...,www.globenewswire.com,0.230660,0.104543,"[Economy - Monetary, Financial Markets]",RIME
70,International Restaurant Management Group ( I...,https://www.globenewswire.com/news-release/202...,20241218T152600,IRMG's portfolio includes over 200 restaurants...,www.globenewswire.com,0.315241,0.0,[Retail & Wholesale],SPGI
21,International Restaurant Management Group ( I...,https://www.globenewswire.com/news-release/202...,20241218T152600,IRMG's portfolio includes over 200 restaurants...,www.globenewswire.com,0.315241,0.04097,[Retail & Wholesale],XCH
69,International Restaurant Management Group ( I...,https://www.globenewswire.com/news-release/202...,20241218T152600,IRMG's portfolio includes over 200 restaurants...,www.globenewswire.com,0.315241,-0.023344,[Retail & Wholesale],TSLA


In [50]:
df_2 = df_2.rename({'summary': 'description', 'time_published': 'published_at', 'source_domain': 'source'}, axis=1)

In [51]:
df_2.head()

,title,url,published_at,description,source,overall_sentiment_score,ticker_sentiment,industry,ticker
0,Crystal Bridges Museum of American Art Awards ...,https://www.benzinga.com/pressreleases/24/12/g...,20241218T160000,"BENTONVILLE, Ark, Dec. 18, 2024 ( GLOBE NEWSWI...",www.benzinga.com,0.420169,0.202252,[Manufacturing],TSN
1,Linda Itskovitz Joins Technology Credit Union'...,https://www.benzinga.com/pressreleases/24/12/g...,20241218T160000,"SAN JOSE, Calif., Dec. 18, 2024 ( GLOBE NEWSWI...",www.benzinga.com,0.404269,0.07783,[Technology],MTTR
2,ThoughtSpot Appoints Francois Lopitaux to Lead...,https://www.benzinga.com/pressreleases/24/12/g...,20241218T160000,"MOUNTAIN VIEW, Calif., Dec. 18, 2024 ( GLOBE N...",www.benzinga.com,0.389167,0.299356,[Technology],SPGI
3,What's Going On With Polyrizon Shares Wednesda...,https://www.benzinga.com/24/12/42557322/whats-...,20241218T155734,Polyrizon Ltd. PLRZ shares are moving higher a...,www.benzinga.com,0.359258,0.166019,[Financial Markets],VRSK
4,Discover the BRETT ( ETH ) Listing on XT,https://www.benzinga.com/pressreleases/24/12/g...,20241218T155254,"VICTORIA, Seychelles, Dec. 18, 2024 ( GLOBE NE...",www.benzinga.com,0.531296,0.563971,"[Blockchain, Financial Markets]",PLRZ


In [39]:
df_2.to_csv('testfile_2.csv')

In [42]:
stockdata = requests.get('https://api.stockdata.org/v1/news/all?countries=us,ca&languages=en&api_token=xbGGrZ1B4MXQBuQwpdyBRlSKywQkDJaId5On0u6j')

if stockdata.status_code == 200:
    # Parse the response as JSON
    stockdata_response = stockdata.json()

#y = requests.get('https://financialmodelingprep.com/api/v3/stock_news?apikey=05zoMFiu5fONSnvYrLHha92IlRlHBzav')



#l = requests.get('https://eodhd.com/api/news?offset=0&limit=1000&api_token=675cef6476cf85.39742731&fmt=json')





#processing stockdata first 


df_1 = pd.DataFrame(stockdata_response["data"])

df_1.drop(['uuid', 'image_url', 'language', 'entities', 'similar', 'relevance_score', 'snippet', 'keywords'], axis=1, inplace=True)

tickers = []
names = []
scores = []
industries = []

rowcount = 0
# Iterate through the articles
for article in stockdata_response["data"]:
    
    # Check if 'entities' exists and is a list
    if 'entities' in article and article['entities']:
        # Use copy() to duplicate the specific row 
        new_row = df_1.loc[rowcount].copy()
        # Append the new row as many times as there are entities left 
        duplicatedrows = len(article['entities']) - 1
        df_1 = df_1._append([new_row] * duplicatedrows, ignore_index=True)

        for entity in article['entities']:
            tickers.append(entity.get('symbol', None))  # Use .get() to handle missing keys
            names.append(entity.get('name', None))
            scores.append(entity.get('sentiment_score', None))
            industries.append(entity.get('industry', None))
    
    rowcount+=1

df_1['ticker'] = tickers

df_1['names'] = names 

df_1['sentiment score'] = scores

df_1['industry'] = industries 

In [69]:
df_1 = df_1.rename({'sentiment score': 'ticker_sentiment'}, axis=1)

df_1.sort_values('url', inplace=True)

df_1.drop(columns='names', inplace=True)

In [70]:
df_1

,title,description,url,published_at,source,ticker,ticker_sentiment,industry
1,"Jabil pops as cloud, data center boosts Q1 res...",Jabil (JBL) was in focus Wednesday after the e...,https://seekingalpha.com/news/4387698-jabil-po...,2024-12-18T15:07:49.000000Z,seekingalpha.com,JBL,0.52665,Technology
0,Accor expands its footprint in Goa with new lu...,Accor expands its footprint in Goa with new lu...,https://www.investing.com/news/press-releases/...,2024-12-18T15:20:04.000000Z,investing.com,EUXTF,0.65730,Financial Services


In [71]:
#concatenating both the dfs into one df 
df_final = pd.concat([df_1, df_2], ignore_index=True)

In [75]:
df_final.sort_values('published_at', inplace=True)

In [76]:
df_final

,title,description,url,published_at,source,ticker,ticker_sentiment,industry,overall_sentiment_score
0,"Jabil pops as cloud, data center boosts Q1 res...",Jabil (JBL) was in focus Wednesday after the e...,https://seekingalpha.com/news/4387698-jabil-po...,2024-12-18T15:07:49.000000Z,seekingalpha.com,JBL,0.52665,Technology,NaN
1,Accor expands its footprint in Goa with new lu...,Accor expands its footprint in Goa with new lu...,https://www.investing.com/news/press-releases/...,2024-12-18T15:20:04.000000Z,investing.com,EUXTF,0.6573,Financial Services,NaN
50,Market Analysis: Apple And Competitors In Tech...,In today's rapidly changing and fiercely compe...,https://www.benzinga.com/insights/news/24/12/4...,20241218T150030,www.benzinga.com,AVGO,0.206408,"[Earnings, Technology, Financial Markets]",0.252399
51,Decoding Alibaba Gr Hldgs's Options Activity: ...,Investors with a lot of money to spend have ta...,https://www.benzinga.com/insights/options/24/1...,20241218T150030,www.benzinga.com,TRGP,0.100754,"[Earnings, Finance, Financial Markets]",0.192614
109,Decoding Alibaba Gr Hldgs's Options Activity: ...,Investors with a lot of money to spend have ta...,https://www.benzinga.com/insights/options/24/1...,20241218T150030,www.benzinga.com,BCS,0.040517,"[Earnings, Finance, Financial Markets]",0.192614
...,...,...,...,...,...,...,...,...,...
5,What's Going On With Polyrizon Shares Wednesda...,Polyrizon Ltd. PLRZ shares are moving higher a...,https://www.benzinga.com/24/12/42557322/whats-...,20241218T155734,www.benzinga.com,VRSK,0.166019,[Financial Markets],0.359258
2,Crystal Bridges Museum of American Art Awards ...,"BENTONVILLE, Ark, Dec. 18, 2024 ( GLOBE NEWSWI...",https://www.benzinga.com/pressreleases/24/12/g...,20241218T160000,www.benzinga.com,TSN,0.202252,[Manufacturing],0.420169
4,ThoughtSpot Appoints Francois Lopitaux to Lead...,"MOUNTAIN VIEW, Calif., Dec. 18, 2024 ( GLOBE N...",https://www.benzinga.com/pressreleases/24/12/g...,20241218T160000,www.benzinga.com,SPGI,0.299356,[Technology],0.389167
52,Linda Itskovitz Joins Technology Credit Union'...,"SAN JOSE, Calif., Dec. 18, 2024 ( GLOBE NEWSWI...",https://www.benzinga.com/pressreleases/24/12/g...,20241218T160000,www.benzinga.com,GOOG,0.171914,[Technology],0.404269
